# wk4 - Demo-2 - Spark SQL

In [ ]:
# Week 4
# Demo code 2 - SparkSQL

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName('ProgBigData-Spark - wk4 Demo 2') \
        .master('local[*]') \
        .getOrCreate()

In [ ]:
#Define the record structure
from pyspark.sql.types import *

fire_schema = StructType([StructField('CallNumber', IntegerType(), True),
   StructField('UnitID', StringType(), True),
   StructField('IncidentNumber', IntegerType(), True),
   StructField('CallType', StringType(), True),
   StructField('CallDate', StringType(), True),
   StructField('WatchDate', StringType(), True),
   StructField('CallFinalDisposition', StringType(), True),
   StructField('AvailableDtTm', StringType(), True),
   StructField('Address', StringType(), True),
   StructField('City', StringType(), True),
   StructField('Zipcode', IntegerType(), True),
   StructField('Battalion', StringType(), True),
   StructField('StationArea', StringType(), True),
   StructField('Box', StringType(), True),
   StructField('OriginalPriority', StringType(), True),
   StructField('Priority', StringType(), True),
   StructField('FinalPriority', IntegerType(), True),
   StructField('ALSUnit', BooleanType(), True),
   StructField('CallTypeGroup', StringType(), True),
   StructField('NumAlarms', IntegerType(), True),
   StructField('UnitType', StringType(), True),
   StructField('UnitSequenceInCallDispatch', IntegerType(), True),
   StructField('FirePreventionDistrict', StringType(), True),
   StructField('SupervisorDistrict', StringType(), True),
   StructField('Neighborhood', StringType(), True),
   StructField('Location', StringType(), True),
   StructField('RowID', StringType(), True),
   StructField('Delay', FloatType(), True)])

# Use the DataFrameReader interface to read a CSV file
sf_fire_file = "datasets/sf-fire-calls.csv"
fire_df = spark.read.csv(sf_fire_file, header=True, schema=fire_schema)

In [ ]:
fire_df.show(10)

In [ ]:
#Now load the Flight Delays data set
#https://github.com/databricks/LearningSparkV2

## Path to data set
csv_file = "datasets/departuredelays.csv"

#Read and create a temporary view
# Infer schema (note that for larger files you
# may want to specify the schema)
flight_delays_df = (spark.read.format("csv")
.option("inferSchema", "true")
.option("header", "true")
.load(csv_file))
#df.createOrReplaceTempView("us_delay_flights_tbl")

flight_delays_df.show(10)

# 3 Define a TempView for use with SQL

In [ ]:
fire_df.createOrReplaceTempView("fire_tab")

# 4 Using SQL on Spark Dataframe (using TempView)

In [ ]:
#When you execute a SQL query, the result set is returned as a Dataframe
spark.sql("SELECT * FROM fire_tab")

In [ ]:
spark.sql("SELECT * FROM fire_tab").show()

In [ ]:
spark.sql("SELECT IncidentNumber, AvailableDtTm, CallType FROM fire_tab").show()

In [ ]:
spark.sql("SELECT IncidentNumber, AvailableDtTm, CallType FROM fire_tab WHERE CallType = 'Medical Incident'").show()

In [ ]:
#this will give an error message
spark.sql("SELECT IncidentNumber, AvailableDtTm, CallType FROM fire_tab \
           WHERE CallType = 'Medical Incident'").show()

In [ ]:
#Put 3x " around the SQL to allow multi-line statements
spark.sql("""SELECT IncidentNumber, AvailableDtTm, CallType FROM fire_tab
             WHERE CallType = 'Medical Incident'""").show()

In [ ]:
#Put 3x " around the SQL to allow multi-line statements
spark.sql("""SELECT IncidentNumber, AvailableDtTm, CallType FROM fire_tab
             WHERE CallType = 'Medical Incident'""").show(10,truncate=False)

In [ ]:
#Let's transform the previous Notebook queries on this data set into SQL
#What if we want to know how many distinct CallTypes values

#fire_df.select("CallType").where(col("CallType").isNotNull()).distinct().count()

spark.sql("SELECT count(distinct CallType) FROM fire_tab").show()

In [ ]:
#Exploring the data - What are the distinct CallTypes
#fire_df.select("CallType") \
#       .distinct() \
#       .show(10, False) \
#       .orderBy("CallType", ascending=True)

spark.sql("SELECT distinct CallType FROM fire_tab ORDER BY CallType").show(10,truncate=False)
#Try this with and without the order by

In [ ]:
#Aggregations - grounBy, Count, orderBy
#fire_df.select("CallType").where(col("CallType").isNotNull()) \
#       .groupBy("CallType") \
#       .count() \
#       .orderBy("count", ascending=False) \
#       .show(n=10, truncate=False)

spark.sql("SELECT CallType, count(*) as cnt FROM fire_tab GROUP BY CallType ORDER BY cnt DESC").show(10,truncate=False)

### Q: What are the most common Zip Codes accounting for call outs from the Fire Department

In [ ]:
spark.sql("SELECT Zipcode, count(*) as cnt FROM fire_tab GROUP BY Zipcode ORDER BY cnt DESC").show(10,truncate=False)

In [ ]:
#new_fire_df.select(F.sum("NumAlarms"), F.avg("ResponseDelayedinMins"), \
#           F.min("ResponseDelayedinMins"), F.max("ResponseDelayedinMins")) \
#.show()
spark.sql("SELECT sum(NumAlarms), avg(Delay), min(Delay), max(Delay) FROM fire_tab").show()

# 5 Using SQL to Join Dataframes (using TempView)

In [ ]:
#Joining tables
from pyspark.sql import Row

#setup 2 dataframes
#1st a Customer df
cust_df = spark.createDataFrame([
    Row(A=1, County='D', CName='John'),
    Row(A=2, County='C', CName='Patrick'),
    Row(A=3, County='D', CName='Seamus'),
    Row(A=4, County='CN', CName='George'),
    Row(A=5, County='MH', CName='Sean'),
    Row(A=6, County='C', CName='Mary'),
    Row(A=7, County='MN', CName='Ann'),
    Row(A=8, County='MN', CName='Jane'),
    Row(A=9, County='TN', CName='Deirdre'),
    Row(A=10, County='D', CName='Amanda'),
    Row(A=11, County='KY', CName='Richard')
])
cust_df  #spark DF
#setup a County df
county_df = spark.createDataFrame([
    Row(County='D', CountyName='Dublin'),
    Row(County='C', CountyName='Cork'),
    Row(County='CN', CountyName='Cavan'),
    Row(County='MH', CountyName='Meath'),
    Row(County='MN', CountyName='Dublin'),
    Row(County='WW', CountyName='Wicklow'),
    Row(County='G', CountyName='Galway')
])
county_df  #spark DF

In [ ]:
#Create Tables for these DFs
cust_df.createOrReplaceTempView("cust_tab")
county_df.createOrReplaceTempView("county_tab")

In [ ]:
#Left Join
#left_join = cust_df.join(county_df, cust_df.County == county_df.County, how='left').sort(cust_df.A)
#left_join.show()
spark.sql("SELECT * FROM cust_tab a LEFT JOIN county_tab b ON a.County = b.County ").show()

In [ ]:
#Inner Join
#inner_join = cust_df.join(county_df, cust_df.County == county_df.County, how='inner').sort(cust_df.A)
#inner_join.show()

#The nulls have disappeared
#But some data is missing. Why?

spark.sql("SELECT * FROM cust_tab a INNER JOIN county_tab b ON a.County = b.County ").show()

In [ ]:
#Right join
#right_join = cust_df.join(county_df, cust_df.County == county_df.County, how='right').sort(cust_df.A)
#right_join.show()

#We have Nulls again - This time in a different location
#Again a Data Quality issue

spark.sql("SELECT * FROM cust_tab a RIGHT JOIN county_tab b ON a.County = b.County ").show()

In [ ]:
#Outer join
#outer_join = cust_df.join(county_df, cust_df.County == county_df.County, how='outer').sort(cust_df.A)
#outer_join.show()

#We get nulls on both sides
#Data Quality issue

spark.sql("SELECT * FROM cust_tab a FULL OUTER JOIN county_tab b ON a.County = b.County ").show()
#Here we needed to specify  FULL OUTER JOIN

In [ ]:
#spark.sql("SELECT * FROM cust_tab").show()
spark.sql("SELECT County, count(*) FROM cust_tab GROUP BY County").show()

spark.sql("SELECT County, count(*) as cnt, dense_rank() OVER (PARTITION BY County ORDER BY count(*) DESC) FROM cust_tab GROUP BY County").show()

In [ ]:
#Windowing SQL Functions (Analytic SQL Functions)
spark.sql("""SELECT ZipCode, CallType, count(*) as cnt, dense_rank() OVER (PARTITION BY ZipCode ORDER BY count(*) DESC) as drank FROM fire_tab WHERE ZipCode is not null GROUP BY ZipCode, CallType""").show(50, truncate=False)

In [ ]:
spark.sql("""SELECT City, CallType, count(*) as cnt, dense_rank() OVER (PARTITION BY City ORDER BY count(*) DESC) as drank FROM fire_tab WHERE City is not null GROUP BY City, CallType""").show(50, truncate=False)